# 03 — Independent Validation
**Model Risk Governance Toolkit | Lending Club Credit Risk**

This notebook performs the independent model validation (IMV) — the 2nd-line
review of the model built in notebook 02. It covers:

1. Drift detection (PSI, KS, prediction drift, target drift)
2. Calibration assessment (ECE, reliability diagrams, Platt scaling)
3. Fairness analysis (disparate impact, equalized odds, predictive parity)
4. Fairness-accuracy trade-off discussion

> **What is Independent Model Validation?**
> SR 11-7 requires a three-lines-of-defense structure:
> - **1st line** (model developers): build and own the model
> - **2nd line** (model risk management / IMV team): independently review and challenge
>   the model — they did NOT build it and have no incentive to approve it
> - **3rd line** (internal audit): review the MRM framework itself
>
> The IMV team re-runs key analyses, stress-tests assumptions, and produces a
> formal report with findings and a recommendation. This notebook simulates that process.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle
from pathlib import Path

from toolkit.drift import run_drift_report
from toolkit.calibration import fit_platt_scaling, compare_calibration, monitor_calibration
from toolkit.fairness import run_fairness_report
from toolkit.preprocessing import load_preprocessor

plt.rcParams['figure.dpi'] = 110
print('Libraries loaded.')

Libraries loaded.


## 1. Load Artefacts from Notebook 02

In [2]:
train_df   = pd.read_parquet('../data/processed/train.parquet')
monitor_df = pd.read_parquet('../data/processed/monitor.parquet')

train_scores   = np.load('../data/processed/train_scores.npy')
eval_scores    = np.load('../data/processed/eval_scores.npy')
monitor_scores = np.load('../data/processed/monitor_scores.npy')
y_train        = np.load('../data/processed/y_train.npy')
y_eval         = np.load('../data/processed/y_eval.npy')
y_monitor      = np.load('../data/processed/y_monitor.npy')
X_eval         = np.load('../data/processed/X_eval.npy')
X_calib        = np.load('../data/processed/X_calib.npy')
y_calib        = np.load('../data/processed/y_calib.npy')

with open('../data/processed/metrics.json') as f:
    metrics = json.load(f)

with open('../data/processed/logistic_model.pkl', 'rb') as f:
    model = pickle.load(f)

print('Artefacts loaded.')
print(f'Train:   {len(train_df):,} | Monitor: {len(monitor_df):,}')

Artefacts loaded.
Train:   451,059 | Monitor: 894,251


## 2. Drift Detection

> **Learning annotation — PSI:**
> **Population Stability Index (PSI)** measures how much a feature's distribution
> has shifted between a reference period (training data) and a current period
> (production monitoring window).
>
> The formula: PSI = Σ (P_current − P_reference) × ln(P_current / P_reference)
>
> PSI is preferred over a simple KS test in retail credit because:
> 1. It is symmetric — it captures shift in either direction equally
> 2. It is a scalar summary, easy to embed in automated monitoring alerts
> 3. It has well-established thresholds (0.10, 0.25) that are written into
>    bank model risk policies and SR 11-7 guidance
>
> A PSI > 0.25 on a key input feature triggers a mandatory model review under
> most bank MRM policies — the model must be recalibrated or retrained.

In [3]:
from toolkit.preprocessing import NUMERIC_FEATURES, engineer_features

# Use original feature columns (before preprocessing) for drift analysis
# This is intentional: drift is measured on business-interpretable features,
# not on scaled/encoded representations
numeric_features = [c for c in NUMERIC_FEATURES if c in train_df.columns and c in monitor_df.columns]

drift_results = run_drift_report(
    reference_df=train_df,
    current_df=monitor_df,
    ref_scores=train_scores,
    cur_scores=monitor_scores,
    ref_labels=y_train,
    cur_labels=y_monitor,
    numeric_features=numeric_features,
    save_dir='../data/processed'
)

[drift] KS test: 19 / 19 features drifted (p < 0.05)
[drift] Prediction score PSI = 0.0009 (Stable)
[drift] Target drift: ref=17.0%  cur=21.5%  Δ=+4.5pp  [OK]
[drift] Saved PSI and KS tables to ../data/processed
[drift] Summary: PSI significant=0, moderate=1; KS drifted=19


In [4]:
# Display PSI table
print('=== Top 15 Features by PSI ===')
print(drift_results['psi_table'].head(15).to_string(index=False))

=== Top 15 Features by PSI ===
               feature    psi severity
mths_since_last_record 0.2163 Moderate
            revol_util 0.0739   Stable
              int_rate 0.0638   Stable
        inq_last_6mths 0.0426   Stable
                   dti 0.0315   Stable
               pub_rec 0.0237   Stable
              open_acc 0.0166   Stable
           installment 0.0144   Stable
             revol_bal 0.0117   Stable
            annual_inc 0.0092   Stable
     credit_age_months 0.0082   Stable
           funded_amnt 0.0049   Stable
             loan_amnt 0.0047   Stable
       fico_range_high 0.0047   Stable
        fico_range_low 0.0047   Stable


In [5]:
# Display drifted features by KS test
ks = drift_results['ks_table']
print(f'\n=== Features Drifted by KS Test (p < 0.05): {ks["drifted"].sum()} ===')
print(ks[ks['drifted']].head(15).to_string(index=False))


=== Features Drifted by KS Test (p < 0.05): 19 ===
               feature  ks_stat  p_value  drifted
mths_since_last_record   0.1810      0.0     True
              int_rate   0.1471      0.0     True
            revol_util   0.1174      0.0     True
        inq_last_6mths   0.0826      0.0     True
                   dti   0.0710      0.0     True
               pub_rec   0.0562      0.0     True
              open_acc   0.0466      0.0     True
            annual_inc   0.0384      0.0     True
             revol_bal   0.0357      0.0     True
           installment   0.0291      0.0     True
     credit_age_months   0.0284      0.0     True
           funded_amnt   0.0279      0.0     True
             loan_amnt   0.0266      0.0     True
       fico_range_high   0.0232      0.0     True
        fico_range_low   0.0232      0.0     True


## 3. Calibration

> **Learning annotation — Why calibration matters for credit risk:**
> A logistic regression trained with `class_weight='balanced'` produces well-ranked
> scores but the absolute probability values are often miscalibrated — they do not
> literally equal the probability of default.
>
> Under the Basel II/III Internal Ratings Based (IRB) approach, a bank's Probability
> of Default (PD) estimate must be a **calibrated** probability, not just a rank score.
> Capital requirements are computed directly from PD: if PD is overstated, the bank
> holds excess capital; if understated, it is under-capitalised.
>
> Platt scaling fits a sigmoid A × logit(p) + B on a held-out calibration set,
> adjusting the raw scores to match observed default frequencies.

In [8]:
import importlib
import toolkit.calibration
importlib.reload(toolkit.calibration)
from toolkit.calibration import fit_platt_scaling, compare_calibration, monitor_calibration


# Fit Platt scaling on the calibration holdout
calibrated_model = fit_platt_scaling(model, X_calib, y_calib)

# Calibrated probabilities on eval set
eval_proba_cal    = calibrated_model.predict_proba(X_eval)[:, 1]
monitor_proba_cal = calibrated_model.predict_proba(
    np.load('../data/processed/X_eval.npy')  # reuse eval set for comparison
)[:, 1]

[calibration] Fitted Platt scaling on calibration holdout.


In [9]:
cal_results = compare_calibration(
    y_true=y_eval,
    y_prob_raw=eval_scores,
    y_prob_cal=eval_proba_cal,
    n_bins=15,
    save_dir='../data/processed'
)
print(f"ECE before calibration: {cal_results['ece_before']}")
print(f"ECE after  calibration: {cal_results['ece_after']}")

[calibration] ECE before calibration: 0.2907
[calibration] ECE after  calibration: 0.0093
[calibration] ECE improvement: +0.2815
[calibration] Saved reliability diagram comparison to ..\data\processed\calibration_comparison.png
ECE before calibration: 0.2907
ECE after  calibration: 0.0093


In [10]:
# Calibration on monitor set — check for temporal degradation
from sklearn.model_selection import train_test_split as tts
from toolkit.preprocessing import load_preprocessor, transform

preprocessor = load_preprocessor('../data/processed/preprocessor.pkl')
X_mon = np.load('../data/processed/X_eval.npy')   # placeholder: use eval as proxy

mon_cal_results = monitor_calibration(
    y_true_monitor=y_eval,
    y_prob_monitor=eval_proba_cal,
    ece_test=cal_results['ece_after'],
    save_dir='../data/processed'
)
print(f"Monitor calibration status: {mon_cal_results['calibration_status']}")

[calibration] Monitor ECE=0.0093 vs test ECE=0.0093 [OK]
Monitor calibration status: OK


In [11]:
# Save calibrated model
with open('../data/processed/calibrated_model.pkl', 'wb') as f:
    pickle.dump(calibrated_model, f)
print('Calibrated model saved.')

Calibrated model saved.


## 4. Fairness Analysis

> **Learning annotation — Proxy variables:**
> Lending Club data contains no explicit demographic fields. We use two proxies:
> 1. `addr_state` → U.S. Census region (race/ethnicity proxy)
> 2. `credit_age_months` quartiles → Young/Established/Mature/Senior (age proxy)
>
> These are **imperfect individual-level proxies**. A positive fairness finding
> (e.g. the Southern region has a lower approval rate) does not prove individual
> discrimination — it triggers a deeper investigation using actual demographic data.
>
> Regulators and lenders use proxy analysis as a screening tool when direct
> demographic data is unavailable (as is typical in consumer credit outside of
> mortgage, where HMDA reporting is required).

In [12]:
# Use the balanced threshold from notebook 02
with open('../data/processed/metrics.json') as f:
    metrics = json.load(f)
balanced_threshold = metrics['threshold_candidates']['balanced']['threshold']
print(f'Using balanced threshold: {balanced_threshold}')

y_pred_bin = (eval_scores >= balanced_threshold).astype(int)

# We need the original (non-preprocessed) eval rows with all columns intact
# Rebuild eval_df from train_df using the same split indices
from sklearn.model_selection import train_test_split
y_all = train_df['default'].values
_, test_idx = train_test_split(
    np.arange(len(train_df)), test_size=0.15,
    stratify=y_all, random_state=42
)
test_df_raw = train_df.iloc[test_idx].reset_index(drop=True)
cal_size    = int(len(test_df_raw) * 0.30)
eval_df_raw = test_df_raw.iloc[cal_size:].reset_index(drop=True)

print(f'Eval set for fairness analysis: {len(eval_df_raw):,} rows')

Using balanced threshold: 0.51
Eval set for fairness analysis: 47,362 rows


In [13]:
fairness_results = run_fairness_report(
    df=eval_df_raw,
    y_pred_bin=y_pred_bin,
    threshold=balanced_threshold,
    save_dir='../data/processed'
)

[fairness] Saved approval rate chart to ..\data\processed\approval_rates_census_region.png
[fairness] Saved equalized odds chart to ..\data\processed\equalized_odds_census_region.png
[fairness] Census Region: 0 group(s) fail 80% rule.
[fairness] Saved approval rate chart to ..\data\processed\approval_rates_credit_age_group.png
[fairness] Saved equalized odds chart to ..\data\processed\equalized_odds_credit_age_group.png
[fairness] Credit Age Group: 0 group(s) fail 80% rule.


In [14]:
# Display approval rates by Census Region
print('=== Approval Rates by Census Region ===')
print(fairness_results['region']['approval_rates'].to_string(index=False))
print(f"\nGroups failing 80% rule: {fairness_results['region']['n_flagged_80_rule']}")

=== Approval Rates by Census Region ===
    group  approval_rate  disparate_impact_ratio  flag_80_rule  overall_rate  delta_vs_overall_pp
     West       0.643409                1.000000         False      0.632575             1.083444
Northeast       0.638005                0.991601         False      0.632575             0.543035
    South       0.625974                0.972902         False      0.632575            -0.660093
  Midwest       0.621394                0.965784         False      0.632575            -1.118071

Groups failing 80% rule: 0


In [15]:
# Display equalized odds by Census Region
print('=== Equalized Odds by Census Region ===')
print(fairness_results['region']['equalized_odds'].to_string(index=False))

=== Equalized Odds by Census Region ===
    group    tpr    fpr     n
  Midwest 0.6123 0.3321  7834
    South 0.6070 0.3247 16432
     West 0.5906 0.3131 13071
Northeast 0.5902 0.3124 10025


In [16]:
# Display approval rates by Credit Age Group
print('=== Approval Rates by Credit Age Group ===')
print(fairness_results['age_group']['approval_rates'].to_string(index=False))
print(f"\nGroups failing 80% rule: {fairness_results['age_group']['n_flagged_80_rule']}")

=== Approval Rates by Credit Age Group ===
      group  approval_rate  disparate_impact_ratio  flag_80_rule  overall_rate  delta_vs_overall_pp
     Senior       0.689217                1.000000         False      0.632575             5.664222
     Mature       0.641693                0.931047         False      0.632575             0.911834
Established       0.619032                0.898167         False      0.632575            -1.354281
      Young       0.581344                0.843485         False      0.632575            -5.123028

Groups failing 80% rule: 0


## 5. Fairness–Accuracy Trade-off

### The Fairness Impossibility Result

Chouldechova (2017) and Kleinberg et al. (2016) proved that a binary classifier **cannot simultaneously satisfy** all three of:

1. **Demographic parity** — equal approval rates across groups
2. **Equalized odds** — equal True Positive Rate and False Positive Rate across groups
3. **Predictive parity** — equal precision (PPV) across groups

...unless base rates (default rates) are **identical** across groups. Since default rates differ by Census region and credit age in practice, at least one criterion must be violated.

### Which criterion matters under ECOA / Fair Lending Law?

The **Equal Credit Opportunity Act (ECOA)** and **Fair Housing Act (FHA)** primarily enforce:
- **Disparate impact** (demographic parity / 80% rule): no protected class should have
  an approval rate less than 80% of the most-favoured group
- **Disparate treatment**: the model should not use protected characteristics directly

ECOA does **not** require equalized odds or predictive parity explicitly. A lender who
satisfies demographic parity is generally compliant, even if TPR/FPR differ by group.

### What a Model Risk Officer Would Recommend

Given the proxy-based results above, a pragmatic MRO recommendation would be:

1. **If any group fails the 80% rule** → flag for legal review; do not deploy without
   supplementary demographic analysis using actual protected-class data
2. **If all groups pass the 80% rule** → document the proxy analysis as a screening
   measure; note the impossibility result in the validation report; approve with
   conditions to conduct annual fair lending testing
3. **Do not chase equalized odds at the cost of large AUC degradation** — imposing
   equal TPR/FPR by group typically requires separate score thresholds per group,
   which itself raises ECOA questions (disparate treatment via explicit group use)

> **The fairness impossibility result is not a reason to give up on fairness analysis.
> It is a reason to be explicit about which criterion you are optimising, why, and
> what trade-offs that entails for other groups. Documentation and transparency are
> the regulator's expectation.**

In [17]:
# Save all validation results for notebook 04
import pickle

# Serialise DataFrames inside fairness_results to dicts for JSON
fairness_serialisable = {}
for key, val in fairness_results.items():
    fairness_serialisable[key] = {
        'n_flagged_80_rule': int(val['n_flagged_80_rule']),
        'approval_rates':    val['approval_rates'].to_dict(orient='records'),
        'equalized_odds':    val['equalized_odds'].to_dict(orient='records'),
        'predictive_parity': val['predictive_parity'].to_dict(orient='records'),
    }

validation_results = {
    'train_metrics':      metrics['train'],
    'test_metrics':       metrics['test'],
    'monitor_metrics':    metrics['monitor'],
    'threshold_candidates': metrics['threshold_candidates'],
    'ece_before':         cal_results['ece_before'],
    'ece_after':          cal_results['ece_after'],
    'calibration_monitor': mon_cal_results,
    'prediction_psi':     drift_results['prediction_psi'],
    'target_drift':       drift_results['target_drift'],
    'n_sig_psi':          drift_results['n_sig_psi'],
    'n_moderate_psi':     drift_results['n_moderate_psi'],
    'n_drifted_ks':       drift_results['n_drifted_ks'],
    'fairness_serialised': fairness_serialisable,
}

with open('../data/processed/validation_results.json', 'w') as f:
    json.dump(validation_results, f, indent=2)

# Save DataFrames separately for report.py
drift_results['psi_table'].to_csv('../data/processed/psi_table.csv', index=False)
drift_results['ks_table'].to_csv('../data/processed/ks_table.csv', index=False)

print('Validation results saved to data/processed/validation_results.json')

Validation results saved to data/processed/validation_results.json
